In [ ]:
import os, sys

p = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(p):
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        zrok_token = f.read().strip()

if not zrok_token:
    print("❌ Token not found or empty:", p)
    sys.exit(1)

In [ ]:
!pip install kaggle-benchmarks

In [ ]:
# --------------------------
# FastAPI app init
# --------------------------

from fastapi import FastAPI
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()  # allow running uvicorn inside the notebook
app = FastAPI()
print("FastAPI app initialized.")

In [ ]:
# --------------------------
# OpenAI Compatible Endpoints
# --------------------------

from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any, Union
import time
import uuid
from fastapi import Request, Header, HTTPException

# --- Pydantic Models for OpenAI Schema ---

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[Message]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1024
    stream: Optional[bool] = False

class Choice(BaseModel):
    index: int
    message: Message
    finish_reason: str

class ChatCompletionResponse(BaseModel):
    id: str
    object: str = "chat.completion"
    created: int
    model: str
    choices: List[Choice]
    usage: Dict[str, int]

class Model(BaseModel):
    id: str
    object: str = "model"
    created: int = int(time.time())
    owned_by: str = "kaggle-user"

class ModelList(BaseModel):
    object: str = "list"
    data: List[Model]

# --- API Endpoints ---

@app.get("/v1/models", response_model=ModelList)
async def list_models():
    """List available models."""
    return ModelList(data=[
        Model(id="kaggle-llm-v1"),
        Model(id="gpt-3.5-turbo"), # Included for compatibility with default client settings
        Model(id="gpt-4")
    ])

@app.post("/v1/chat/completions", response_model=ChatCompletionResponse)
async def chat_completions(
    request: ChatCompletionRequest,
    authorization: Optional[str] = Header(None)
):
    """Handle chat completions compatible with OpenAI API."""
    
    # 1. Extract API Key (The 'Model Key' you mentioned)
    api_key = None
    if authorization and authorization.startswith("Bearer "):
        api_key = authorization.split(" ")[1]
    
    print(f"[API] Request for model: {request.model}")
    print(f"[API] Key provided: {'Yes' if api_key else 'No'}")

    # 2. Extract User Prompt
    # (Simplistic logic: gets the last user message)
    last_user_message = ""
    for m in reversed(request.messages):
        if m.role == "user":
            last_user_message = m.content
            break
            
    # ------------------------------------------------------------------
    # INFERENCE LOGIC PLACEHOLDER
    # ------------------------------------------------------------------
    # If you have a specific Kaggle library (e.g. google-generativeai)
    # initialized with the 'api_key', call it here.
    # For now, this is a working echo/mock so the endpoint functions.
    
    response_content = f"[Kaggle Notebook] You said: '{last_user_message}'. \n(To connect a real model, update the logic in the '/v1/chat/completions' function)."
    
    # Example if using Google GenAI (Gemini) inside Kaggle:
    # try:
    #     import google.generativeai as genai
    #     if api_key: genai.configure(api_key=api_key)
    #     model = genai.GenerativeModel('gemini-pro')
    #     response = model.generate_content(last_user_message)
    #     response_content = response.text
    # except Exception as e:
    #     response_content = f"Error calling model: {e}"
    # ------------------------------------------------------------------

    # 3. Construct Response
    return ChatCompletionResponse(
        id=f"chatcmpl-{uuid.uuid4()}",
        created=int(time.time()),
        model=request.model,
        choices=[
            Choice(
                index=0,
                message=Message(role="assistant", content=response_content),
                finish_reason="stop"
            )
        ],
        usage={
            "prompt_tokens": len(last_user_message),
            "completion_tokens": len(response_content),
            "total_tokens": len(last_user_message) + len(response_content)
        }
    )

print("OpenAI compatible endpoints added: /v1/models, /v1/chat/completions")

In [ ]:
# Download zrok v1.1.3 (latest known release used previously)
!wget -q https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz -O /tmp/zrok.tar.gz
!tar -xzf /tmp/zrok.tar.gz -C /tmp
!mv /tmp/zrok /kaggle/working/zrok
!chmod +x /kaggle/working/zrok
!ls -l /kaggle/working/zrok

In [ ]:
# Enable zrok using the token we read at the top (headless)
!/kaggle/working/zrok enable --headless "$zrok_token"

In [ ]:
import uvicorn

# Start uvicorn in a background thread so we can continue running cells
def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_uvicorn, daemon=True).start()
print("uvicorn started on 0.0.0.0:8000")

In [ ]:
import subprocess
import re
import time

def start_zrok_tunnel(port=8000):
    # Start the tunnel process in headless mode
    proc = subprocess.Popen([
        "/kaggle/working/zrok", "share", "public", f"localhost:{port}", "--headless"
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Give it a few seconds to create the tunnel
    time.sleep(3)

    # Fetch agent status to show the public URL
    try:
        status_proc = subprocess.run([
            "/kaggle/working/zrok", "agent", "status"
        ], capture_output=True, text=True, timeout=5)
        print("zrok agent status:\n", status_proc.stdout)
    except Exception as e:
        print("Could not get agent status:", e)

    return proc

# Start the tunnel
tunnel_process = start_zrok_tunnel(8000)
print("Zrok tunnel started (process returned). Check agent status output above for public URL.")

In [ ]:
!/kaggle/working/zrok overview || true

In [ ]:
import time

print("Server and zrok tunnel are running. Keeping the notebook alive (CTRL+C to stop)...")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down.")
